# Research Notebook for PISA question and answers on different languages

## Libraries

In [1]:
import os, getpass

In [2]:
from openai import OpenAI

In [3]:
# --- 0) Setup: imports & config
import os, re, time, json, math, random
from typing import Dict, List, Tuple, Any, Optional
import pandas as pd
import time

In [4]:
from tqdm import tqdm

## Configuration

In [5]:
BASE_URL = "https://ri-delta.ai/"  # <- replace with your portal URL root
MODEL_GPT = "gpt-5"                    # <- replace with exact model id if different
MODEL_CLAUDE = "claude-sonnet-4"
MODEL_GEMINI = "gemini-2.5-pro"

In [6]:
os.environ["LLM_API_KEY"] = "sk-8b736f91707c4d1cb2042dd74c0c647b"
client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=BASE_URL + "/api"    # e.g., https://llm.company.com/v1
)

In [7]:
SHEET_ID = "1QVPzB7uMwqJ6jCsHkwIILnXvDQIycpqkcV3bkiDpzyQ"
WORKSHEET_NAME = "dataset"  # change if needed
RESULTS_CSV = "llm_eval_results.csv"
SAMPLE_PER_LANGUAGE = 1     # 1 per language
MAX_LANGUAGES = 5          # 10 languages total
# LLM_TEMPERATURE = 0.2
# LLM_MAX_TOKENS = 256
SEED = 42

random.seed(SEED)

## Load data from Google Sheet

In [413]:
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={WORKSHEET_NAME}"
try:
    df = pd.read_csv(csv_url)
except Exception as e:
    raise RuntimeError(
        "Failed to read the Google Sheet via CSV export. "
        "Make sure the sheet is shared as 'Anyone with the link can view', "
        f"ID is correct, and tab name matches. Underlying error: {e}"
    )

expected_cols = {
    "qid","language","question","context","options","gold",
    "answer_type","category","difficulty","rationale","source"
}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Your sheet is missing columns: {sorted(missing)}")

In [414]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   qid          139 non-null    object
 1   language     139 non-null    object
 2   question     139 non-null    object
 3   context      139 non-null    object
 4   options      139 non-null    object
 5   gold         139 non-null    object
 6   answer_type  139 non-null    object
 7   category     139 non-null    object
 8   difficulty   139 non-null    object
 9   rationale    60 non-null     object
 10  source       139 non-null    object
dtypes: object(11)
memory usage: 12.1+ KB


## Normalize & sample

In [415]:
df["language"] = df["language"].astype(str).str.strip()
lang_groups = []
for lang, sub in df.groupby("language", sort=True):
    lang_groups.append(sub.iloc[:SAMPLE_PER_LANGUAGE])

In [416]:
sampled = pd.concat(lang_groups, ignore_index=True).iloc[:MAX_LANGUAGES]
if sampled.empty:
    raise ValueError("No rows selected. Check your data.")

In [417]:
print(f"Selected {len(sampled)} rows across {sampled['language'].nunique()} languages.")
display(sampled[["qid","language","question","gold"]])

Selected 5 rows across 5 languages.


,qid,language,question,gold
0,q0006,Albanian,Çfarë përmendin shkencëtarët në artikull për t...,B
1,q0001,Amharic_Machine,"Like nga be suannin koko nun wa’n, mɔ be kan w...",C
2,q0001,Arabic,إذا قررتْ دانا شراء السيارة (د) وباعتها بعد ثل...,C
3,q0001,Baoule_Machine,?Sɛ Tania fa ajalɛ kɛ ɔ́ tó loto D naan ɔ́ yó ...,C
4,q0001,Chinese,如果譚雅決定購買汽車 D 並於三年後在保持良好狀態下轉售，那麼這輛汽車的大約轉售價格將是多少...,C


## Parse options

In [ ]:
import json
import re
import pandas as pd
from typing import Any, Dict

def parse_options(raw: str) -> Dict[str, Any]:
    """
    Parse multiple-choice options from a JSON-encoded string.

    Supported formats
    -----------------
    1) Simple labeled strings (original format):
        ["A) 1575", "B) 8925", "C) 9000", "D) 9975"]

        -> {"A": "1575", "B": "8925", "C": "9000", "D": "9975"}

    2) List of dicts with labels mapping to lists of tokens:
        [
          {"A": ["India", "Colombia"]},
          {"B": ["India", "Armenia"]},
          {"C": ["Panama", "Colombia"]},
          {"D": ["Kazakhstan", "Colombia"]}
        ]
    """
    if pd.isna(raw):
        raise ValueError("Options are empty")

    # Load JSON
    try:
        items = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format for options: {e}")

    if not isinstance(items, list):
        raise ValueError("Expected a JSON list at top level")

    # Case 1: list of strings -> original behavior
    if all(isinstance(item, str) for item in items):
        options: Dict[str, Any] = {}
        for item in items:
            # Match patterns like "A) text", "B. text", or "C: text"
            if not isinstance(item, str):
                raise ValueError(f"Option is not a string: {item}")
            match = re.match(r"^\s*([A-Z])[\)\.\:]\s*(.+)$", item.strip())
            if match:
                label, text = match.groups()
                options[label.upper()] = text.strip()
            else:
                # Fallback: assign next available letter automatically
                next_label = chr(ord('A') + len(options))
                options[next_label] = item.strip()
        return options

    # Case 2: list of dicts like [{"A": [...]}, {"B": [...]}]
    if all(isinstance(item, dict) for item in items):
        options: Dict[str, Any] = {}
        for idx, d in enumerate(items):
            if len(d) != 1:
                raise ValueError(
                    f"Each dict must have exactly one key (label). Problem at index {idx}: {d}"
                )
            (label_raw, value) = next(iter(d.items()))
            if not isinstance(label_raw, str):
                raise ValueError(f"Label must be a string, got: {label_raw}")

            label = label_raw.strip().upper()
            if not re.fullmatch(r"[A-Z]", label):
                raise ValueError(f"Invalid option label '{label_raw}' at index {idx}")

            # Accept list or scalar; normalize scalars into single-element lists if needed
            if isinstance(value, list):
                options[label] = value
            else:
                options[label] = [value]

        return options

    # If we reach here, the list is mixed or has unsupported types
    raise ValueError(
        "Unsupported options format: expected list of strings or list of single-key dicts"
    )


In [419]:
test = parse_options('["A) 2018", "B) 2019", "C) 2020", "D) 2021"]')
print(test)

{'A': '2018', 'B': '2019', 'C': '2020', 'D': '2021'}


In [420]:
options_block = "\n".join([f"{k}. {v}" for k,v in test.items()])
print(options_block)

A. 2018
B. 2019
C. 2020
D. 2021


## Build prompt

In [421]:
def build_prompt(row: pd.Series, options: Dict[str,str]) -> str:
    """
    Builds a language-agnostic but context-aware prompt for MCQ.
    """
    options_block = "\n".join([f"{k}. {v}" for k,v in options.items()])
    # You can adapt language instruction if you want the rationale in the same language:
    lang = str(row["language"]).strip()

    return (
        # f"You are answering a multiple-choice question. \n"
        # f"Return ONLY the chosen option letter (A, B, C, B). \n"
        f"{row['context']}\n"
        f"{row['question']}\n"
        f"{options_block}\n\n"
        # "Reply format STRICTLY:\n" 
        # "<LETTER>\n" 
        # "IMPORTANT: Respond with ONLY the letter of the correct option (A, B, C, D). \n"
        # "No words, no punctuation, no explanation.\n"
        # "Example of valid reply: A\n"
    )

In [422]:
answer_letter_regex = re.compile(r"<\s*([A-Z])\s*[.)]?\s*>")

def extract_letter(text: str, valid_letters: List[str]) -> str:
    """
    Extract the first single-letter A-Z token that is in valid_letters.
    """
    if not text:
        return ""
    # First line is the letter per our format; but still be defensive:
    first_line = text.splitlines()[0].strip().rstrip(".)").upper()
    # If first line is a single valid letter, use it
    if len(first_line) == 1 and first_line.upper() in valid_letters:
        return first_line.upper()
    # Else find any A-Z token
    m = answer_letter_regex.search(text.upper())
    if m and m.group(1) in valid_letters:
        return m.group(1)
    return ""

## LLM call wrapper

In [423]:
def llm_completion(
    prompt: str,
    model: Optional[str] = None,
    # temperature: float = 0.2,
    max_tokens: Optional[int] = None,
    max_completion_tokens: Optional[int] = None,
    max_output_tokens: Optional[int] = None,
    system_prompt: str="Reply format: <LETTER>", # "Reply format: <LETTER>",
    # system_prompt: str="You are a test-taking assistant. Respond ONLY with the letter of the correct answer (A, B, C, or D). Never provide explanations.",
    # system_prompt: str = "IMPORTANT: Answer ONLY with the letter of the correct option (A, B, C, D). Do NOT write anything else.",
    # system_prompt: str = "You are a helpful assistant that answers multiple-choice questions. Reply format: <LETTER>.",
    retries: int = 2,
    backoff_seconds: float = 1.5,
    **kwargs,
) -> str:
    """
    Call an LLM via chat.completions and return text.
    - `prompt`: user content (string)
    - `model`: overrides global MODEL_NAME if provided
    - `temperature`, `max_tokens`: usual decoding controls
    - `system_prompt`: system role content
    - `retries`: retry on transient errors
    - `backoff_seconds`: base backoff between retries
    - `**kwargs`: forwarded to client.chat.completions.create (e.g., stop, seed)
    """
    mdl = model or MODEL_GPT
    last_err = None

    for attempt in range(retries + 1):
        try:
            # Build arguments dynamically — include only if provided
            call_args = dict(
                model=mdl,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
                stream=False
            )

            # ---- attach token limits if provided ----
            if max_tokens is not None:
                call_args["max_tokens"] = max_tokens
            if max_completion_tokens is not None:
                call_args["max_completion_tokens"] = max_completion_tokens
            if max_output_tokens is not None:
                call_args["max_output_tokens"] = max_output_tokens

            # Merge other kwargs (e.g., stop, seed, etc.)
            for k, v in kwargs.items():
                if k not in call_args:
                    call_args[k] = v

            # Actual model call
            response = client.chat.completions.create(**call_args)
            if response is None:
                return "No response from model."
            elif not hasattr(response, "choices") or not response.choices:
                return "Response object missing 'choices'."
            else:
                return(response.choices[0].message.content or "").strip()

        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(backoff_seconds * (attempt + 1))
            else:
                raise

## Evaluation loop

In [424]:
def eval_rows(
        rows: pd.DataFrame, 
        cycle: int,
        model_tag: str,
        model_name: str, 
        results_path: str = RESULTS_CSV,
        sleep_s: float = 0.0, 
        retries: int = 2
    ) -> pd.DataFrame:
    results = []
    file_exists = os.path.exists(results_path)
    for i, row in tqdm(rows.iterrows(), total=len(rows), desc=f"Evaluating {model_tag} / {model_name}, cycle {cycle}"):
        qid = row["qid"]
        lang = row["language"]
        gold = str(row["gold"]).strip().upper()

        # Parse options
        try:
            opts = parse_options(row["options"])
        except Exception as e:
            row_result = {
                "qid": qid,
                "language": lang,
                "pred": "",
                "gold": gold,
                "is_correct": False,
                "error": f"OptionsParseError: {e}",
                "raw": "",
                "question": row["question"],
                "options_json": json.dumps(opts if 'opts' in locals() else {}, ensure_ascii=False),
                "model_tag": model_tag,
                "model_name": model_name,
                "cycle": cycle,
            }
            results.append(row_result)

            # Save immediately
            pd.DataFrame([row_result]).to_csv(
                results_path,
                mode="a",
                header=not file_exists,
                index=False,
                encoding="utf-8"
            )
            file_exists = True
            continue

        valid_letters = sorted(list(opts.keys()))
        prompt = build_prompt(row, opts)

        # call model with simple retry
        raw = ""
        err = ""
        for attempt in range(retries + 1):
            try:
                raw = llm_completion(prompt, model = model_name)
                break
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))
                else:
                    raw = ""
        pred = extract_letter(raw, valid_letters)
        is_correct = (pred == gold)

        row_result = {
            "qid": qid,
            "language": lang,
            "pred": pred,
            "gold": gold,
            "is_correct": bool(is_correct),
            "error": err,
            "raw": raw,
            "question": row["question"],
            "options_json": json.dumps(opts, ensure_ascii=False),
            "model_tag": model_tag,
            "model_name": model_name,
            "cycle": cycle,
        }

        results.append(row_result)

        # *** Save this row immediately ***
        pd.DataFrame([row_result]).to_csv(
            results_path,
            mode="a",
            header=not file_exists,
            index=False,
            encoding="utf-8"
        )
        file_exists = True

        if sleep_s > 0:
            time.sleep(sleep_s)
    return pd.DataFrame(results)

## Execution and results

In [436]:
MODELS_TO_TEST = [
    ("GPT", MODEL_GPT),        # (tag, model_name_for_API)
    # ("Claude", MODEL_CLAUDE),
    # ("Gemini", MODEL_GEMINI)
]

In [437]:
filtered_df = df[df["language"] == "Baoule_Machine"]
sampled = filtered_df

In [438]:
N_CYCLES = 1  # repeat the same question to the same LLM

In [439]:
if os.path.exists(RESULTS_CSV):
    existing_results = pd.read_csv(RESULTS_CSV)
    print(f"Loaded existing results from {RESULTS_CSV}: {len(existing_results)} rows")
else:
    existing_results = pd.DataFrame()
    print("No existing results file found. Starting fresh.")

for tag, model_name in MODELS_TO_TEST:
    print(f"\nEvaluating {tag} -> {model_name}")
    for cycle in range(1, N_CYCLES + 1):
        # Determine which qids are already done for this (tag, model_name, cycle)
        if existing_results.empty:
            # Nothing done yet at all
            rows_to_eval = sampled.copy()
        else:
            # Subset only rows already done for this (tag, model_name, cycle)
            subset = existing_results[
                (existing_results["model_tag"] == tag) &
                (existing_results["model_name"] == model_name) &
                (existing_results["cycle"] == cycle)
            ]

            if subset.empty:
                # No rows done yet for this model+cycle
                rows_to_eval = sampled.copy()
            else:
                # Use (qid, language) pairs as the key — more robust than qid alone
                done_pairs = set(zip(subset["qid"], subset["language"]))

                mask = ~sampled.apply(
                    lambda r: (r["qid"], r["language"]) in done_pairs,
                    axis=1
                )
                rows_to_eval = sampled[mask]

        if rows_to_eval.empty:
            print(f"  • cycle {cycle}/{N_CYCLES}: already complete, skipping")
            continue

        print(f"  • cycle {cycle}/{N_CYCLES}: evaluating {len(rows_to_eval)} questions")
        t0 = time.time()

        df_new = eval_rows(
            rows_to_eval,
            model_tag=tag,
            model_name=model_name,
            cycle=cycle,
            results_path=RESULTS_CSV
        )

        elapsed = time.time() - t0
        print(f"    Done in {elapsed:.1f}s, newly evaluated {len(df_new)} rows.")

        # Update in-memory copy so subsequent cycles/ models can see freshly written rows
        existing_results = pd.concat([existing_results, df_new], ignore_index=True)

Loaded existing results from llm_eval_results.csv: 20 rows

Evaluating GPT -> gpt-5
  • cycle 1/1: evaluating 10 questions


Evaluating GPT / gpt-5, cycle 1: 100%|██████████| 10/10 [03:12<00:00, 19.28s/it]

    Done in 192.8s, newly evaluated 10 rows.


In [440]:
res_df = pd.read_csv(RESULTS_CSV)
print(f"\nTotal results loaded: {len(res_df)}")


Total results loaded: 30


In [441]:
overall_by_model = (
    res_df.groupby(["model_tag","model_name"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by model:")
display(overall_by_model)


Overall accuracy by model:


,model_tag,model_name,accuracy
0,GPT,gpt-5,0.766667


In [442]:
overall_by_lang = (
    res_df.groupby(["language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by lang:")
display(overall_by_lang)


Overall accuracy by lang:


,language,accuracy
0,Amharic_Machine,0.80
1,Baoule_Machine,0.75


In [443]:
by_model_lang = (
    res_df.groupby(["model_tag","model_name","language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["model_tag","language"])
)
print("\nAccuracy by model & language:")
display(by_model_lang)


Accuracy by model & language:


,model_tag,model_name,language,accuracy
0,GPT,gpt-5,Amharic_Machine,0.80
1,GPT,gpt-5,Baoule_Machine,0.75


In [444]:
by_question = (
    res_df.groupby(["model_tag","model_name","language","question"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["accuracy"])
)

# print("\nAccuracy by question:")
# display(by_question)

accuracy_counts_total = (
    by_question["accuracy"]
    .value_counts()
    .rename_axis("accuracy")
    .reset_index(name="count")
    .sort_values("accuracy")
)

print("\nCount of total questions by accuracy:")
display(accuracy_counts_total)

accuracy_counts = (
    by_question
    .groupby(["model_tag", "model_name", "accuracy"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["model_tag", "model_name", "accuracy"])
)

print("\nCount of questions by accuracy:")
display(accuracy_counts)


Count of total questions by accuracy:


,accuracy,count
1,0.0,4
2,0.5,1
0,1.0,15



Count of questions by accuracy:


,model_tag,model_name,accuracy,count
0,GPT,gpt-5,0.0,4
1,GPT,gpt-5,0.5,1
2,GPT,gpt-5,1.0,15


In [445]:
def safe_acc(s):
    return float('nan') if s.empty else s.mean()
overall_acc = safe_acc(res_df["is_correct"])
print(f"\nCombined overall accuracy on {len(res_df)} items: {overall_acc:.3f}")


Combined overall accuracy on 30 items: 0.767


In [446]:
for tag, _ in MODELS_TO_TEST:
    out_path = f"llm_eval_results__{tag}.csv"
    res_df.query("model_tag == @tag").to_csv(out_path, index=False)
    print(f"Saved {tag} results to: {out_path}")

# (Optional) quick peek
display(res_df.head())

Saved GPT results to: llm_eval_results__GPT.csv


,qid,language,pred,gold,is_correct,error,raw,question,options_json,model_tag,model_name,cycle
0,q0001,Amharic_Machine,C,C,True,NaN,C,ታኒያ መኪና ዲ ለመግዛት ከወሰነ እና ከሶስት አመት በኋላ በጥሩ ሁኔታ ለ...,"{""A"": ""1575"", ""B"": ""8925"", ""C"": ""9000"", ""D"": ""...",GPT,gpt-5,1
1,q0002,Amharic_Machine,C,C,True,NaN,C,"ይህ የሽያጭ አዝማሚያ ከቀጠለ, በአምሳያው መሠረት የተሸጡ ዲቪዲዎች ቁጥር...","{""A"": ""2018"", ""B"": ""2019"", ""C"": ""2020"", ""D"": ""...",GPT,gpt-5,1
2,q0003,Amharic_Machine,B,B,True,NaN,B,በጭነት መኪና ውስጥ ሊገጣጠሙ የሚችሉት ከፍተኛው መካከለኛ ሳጥኖች ስንት ...,"{""A"": ""320"", ""B"": ""128"", ""C"": ""96"", ""D"": ""64""}",GPT,gpt-5,1
3,q0004,Amharic_Machine,D,D,True,NaN,D,በአማካይ ፕላኔት ኔፕቱን ከፀሐይ ስንት ሚሊዮን ኪሎ ሜትር ይርቃል?,"{""A"": ""180 ሚሊዮን ኪሜ"", ""B"": ""450 ሚልዮን ኪሜ"", ""C"": ...",GPT,gpt-5,1
4,q0005,Amharic_Machine,B,A,False,NaN,B,በውድድር ዘመኑ ካስመዘገበው አማካይ የአሸናፊነት ልዩነት አንፃር ቡድኑ በ...,"{""A"": ""አዎ"", ""B"": ""አይደለም""}",GPT,gpt-5,1
